# RDS sin validación

Exporta los parámetros de una BO sin entrenar ni evaluar modelos.
Cada archivo `ensembles/man_val/<id_validacion>.rds` tiene su fila correspondiente
en `validaciones_manuales.csv`, con el mismo ID secuencial.

Se conserva el esquema del CSV: `ganancia` queda vacía,
`cantidad_registros_validacion` es cero y `meses_validacion` queda vacío.
`benchmark_original` identifica aquí la carpeta de la BO de origen.

Ejecutar las partes comunes y luego la sección del caso que se quiere exportar.
Los campos de documentación se pueden editar antes de guardar.

# Librerías

In [1]:
import os
import numpy as np
import pandas as pd
import yaml
import rdata
from IPython.display import display

# URLs

In [2]:
BASE_URL = os.path.join(
    os.path.expanduser("~"),
    "code", "DMEyF", "dmeyf2026"
)

DATA_FOLDER = os.path.join(BASE_URL, "entregas", "dataset")
EXP_URL = os.path.join(BASE_URL, "DATA", "EXP")
carpeta_man_val = os.path.join(BASE_URL, "ensembles", "man_val")
archivo_validaciones = os.path.join(carpeta_man_val, "validaciones_manuales.csv")

# Guardado común

El ID se obtiene del registro existente. El RDS contiene una lista de parámetros
individuales, formato que ya admiten las notebooks de entrega.
Se agrega una fila al CSV respetando el orden de sus columnas y conservando sus filas anteriores.

In [3]:
def guardar_rds_sin_validacion(params_completos, registro_validacion):
    existe_archivo = os.path.exists(archivo_validaciones)
    if existe_archivo:
        validaciones_previas = pd.read_csv(archivo_validaciones)
        id_validacion = int(validaciones_previas["id_validacion"].max()) + 1
    else:
        id_validacion = 1

    registro_validacion = pd.DataFrame([{
        "id_validacion": id_validacion,
        **registro_validacion,
    }])
    if existe_archivo:
        registro_validacion = registro_validacion.reindex(columns=validaciones_previas.columns)

    os.makedirs(carpeta_man_val, exist_ok=True)
    archivo_rds = os.path.join(carpeta_man_val, f"{id_validacion}.rds")
    rdata.write_rds(archivo_rds, params_completos)

    registro_validacion.to_csv(
        archivo_validaciones,
        mode="a" if existe_archivo else "w",
        header=not existe_archivo,
        index=False,
    )

    print("RDS:", archivo_rds)
    return registro_validacion

# Caso 1 — BO con etiquetas reales para predecir julio

Modelo entrenado con marzo a junio, utilizado para generar las probabilidades de julio.
Se exportan los mejores HP de esta BO y se conserva su semilla original.
Debe ser el mismo modelo cuyas probabilidades utilizó la BO con pseudoetiquetas.

## Experimento de origen

In [4]:
experimento_origen = "LAG-010"  # editar si el nombre de la carpeta es distinto

## Parámetros del modelo

In [5]:
EXPERIMENT_URL = os.path.join(EXP_URL, experimento_origen)
archivo_parametros = os.path.join(EXPERIMENT_URL, "PARAM.yml")

with open(archivo_parametros, "r", encoding="utf-8") as archivo:
    PARAM_BO = yaml.safe_load(archivo)

params_completos = {
    **PARAM_BO["lgbm"]["param_fijos"],
    **PARAM_BO["out"]["lgbm"]["mejores_hiperparametros"],
}

# Mismo ajuste para entrenamiento completo que en la validación manual.
params_completos["min_data_in_leaf"] = int(np.rint(
    params_completos["min_data_in_leaf"] /
    PARAM_BO["trainingstrategy"]["undersampling"]
))

print("Experimento:", experimento_origen)
print("Semilla del modelo:", params_completos["seed"])
params_completos

Experimento: LAG-010
Semilla del modelo: 230047


{'boosting': 'gbdt',
 'objective': 'binary',
 'metric': 'average_precision',
 'feature_pre_filter': False,
 'first_metric_only': False,
 'boost_from_average': True,
 'force_row_wise': True,
 'deterministic': True,
 'verbosity': -100,
 'seed': 230047,
 'max_depth': -1,
 'min_gain_to_split': 0.0,
 'lambda_l1': 0.0,
 'lambda_l2': 0.0,
 'max_bin': 31,
 'bagging_fraction': 1.0,
 'pos_bagging_fraction': 1.0,
 'neg_bagging_fraction': 1.0,
 'is_unbalance': False,
 'scale_pos_weight': 1.0,
 'drop_rate': 0.1,
 'max_drop': 50,
 'skip_drop': 0.5,
 'extra_trees': False,
 'early_stopping': 0,
 'min_data_in_leaf': 0,
 'feature_fraction': 0.27641343005495034,
 'num_iterations': 4000,
 'num_leaves': 166,
 'min_sum_hessian_in_leaf': 0.1,
 'learning_rate': 0.011685329568129043}

## Documentación del registro

In [6]:
# Campos de documentación editables.
dataset_file = "competencia_01_ternaria_lf_k6_perdida4.csv"
train_final = [202103, 202104, 202105, 202106]
benchmark_original = experimento_origen

columnas_excluir = [
    "ternaria",
    "ternaria_lag1",
    "clase01",
    "fold",
    "azar",
    "training",
    "cprestamos_personales",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "mprestamos_personales",
    "mprestamos_personales_lag1",
    "mprestamos_personales_delta1",
]

# Se registra la semilla conservada en el RDS; no se ejecutan entrenamientos.
SEEDS = [params_completos["seed"]]

# El dataset se lee solamente para contar registros.
dataset_url = os.path.join(DATA_FOLDER, dataset_file)
dataset = pd.read_csv(dataset_url, usecols=["foto_mes"])

registro_validacion = {
    "ganancia": np.nan,
    "seeds": ",".join(map(str, SEEDS)),
    "dataset": dataset_file,
    "cantidad_registros_totales": len(dataset),
    "cantidad_registros_entrenamiento": int(dataset["foto_mes"].isin(train_final).sum()),
    "cantidad_registros_validacion": 0,
    "meses_entrenamiento": ",".join(map(str, train_final)),
    "meses_validacion": "",
    "columnas_excluidas": ",".join(columnas_excluir),
    "benchmark_original": benchmark_original,
}

display(pd.DataFrame([registro_validacion]))

,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
0,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,654066,0,"202103,202104,202105,202106",,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-010


## Guardar RDS y registrar

Ejecutar una vez por exportación. Genera un nuevo ID, su RDS y una fila en el CSV.

In [7]:
registro_guardado = guardar_rds_sin_validacion(params_completos, registro_validacion)
display(registro_guardado)

RDS: /home/marco/code/DMEyF/dmeyf2026/ensembles/man_val/52.rds


,id_validacion,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
0,52,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,654066,0,"202103,202104,202105,202106",,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-010


# Caso 2 — BO con pseudoetiquetas de julio para predecir agosto

Modelo final entrenado con marzo a julio: etiquetas reales de marzo a junio y BAJA+1 de julio;
BAJA+2 estimadas en julio.

Se exportan los mejores HP de la segunda BO. No se genera el dataset con pseudoetiquetas
ni se evalúa agosto en esta notebook. `prob_corte` permanece en `resumen.md` y `BO_log.txt`
del experimento, para ingresarlo como variable global en la notebook de entrega.

## Experimento de origen

In [8]:
experimento_origen = "LAG-PSEUDO-001"  # editar si el nombre de la carpeta es distinto

## Parámetros del modelo

In [9]:
EXPERIMENT_URL = os.path.join(EXP_URL, experimento_origen)
archivo_parametros = os.path.join(EXPERIMENT_URL, "PARAM.yml")

with open(archivo_parametros, "r", encoding="utf-8") as archivo:
    PARAM_BO = yaml.safe_load(archivo)

params_completos = {
    **PARAM_BO["lgbm"]["param_fijos"],
    **PARAM_BO["out"]["lgbm"]["mejores_hiperparametros"],
}

# Mismo ajuste para entrenamiento completo que en la validación manual.
params_completos["min_data_in_leaf"] = int(np.rint(
    params_completos["min_data_in_leaf"] /
    PARAM_BO["trainingstrategy"]["undersampling"]
))

print("Experimento:", experimento_origen)
print("Semilla del modelo:", params_completos["seed"])
params_completos

Experimento: LAG-PSEUDO-001
Semilla del modelo: 230047


{'boosting': 'gbdt',
 'objective': 'binary',
 'metric': 'average_precision',
 'feature_pre_filter': False,
 'first_metric_only': False,
 'boost_from_average': True,
 'force_row_wise': True,
 'deterministic': True,
 'verbosity': -100,
 'seed': 230047,
 'max_depth': -1,
 'min_gain_to_split': 0.0,
 'lambda_l1': 0.0,
 'lambda_l2': 0.0,
 'max_bin': 31,
 'bagging_fraction': 1.0,
 'pos_bagging_fraction': 1.0,
 'neg_bagging_fraction': 1.0,
 'is_unbalance': False,
 'scale_pos_weight': 1.0,
 'drop_rate': 0.1,
 'max_drop': 50,
 'skip_drop': 0.5,
 'extra_trees': False,
 'early_stopping': 0,
 'min_data_in_leaf': 0,
 'feature_fraction': 0.3847092908193044,
 'num_iterations': 3668,
 'num_leaves': 204,
 'min_sum_hessian_in_leaf': 0.00010000000000000009,
 'learning_rate': 0.025}

## Documentación del registro

In [10]:
# Campos de documentación editables.
dataset_file = "competencia_01_ternaria_lf_k6_perdida4.csv"
train_final = [202103, 202104, 202105, 202106, 202107]
benchmark_original = experimento_origen

columnas_excluir = [
    "ternaria",
    "ternaria_lag1",
    "clase01",
    "fold",
    "azar",
    "training",
    "cprestamos_personales",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "mprestamos_personales",
    "mprestamos_personales_lag1",
    "mprestamos_personales_delta1",
]

# Se registra la semilla conservada en el RDS; no se ejecutan entrenamientos.
SEEDS = [params_completos["seed"]]

# El dataset se lee solamente para contar registros.
dataset_url = os.path.join(DATA_FOLDER, dataset_file)
dataset = pd.read_csv(dataset_url, usecols=["foto_mes"])

registro_validacion = {
    "ganancia": np.nan,
    "seeds": ",".join(map(str, SEEDS)),
    "dataset": dataset_file,
    "cantidad_registros_totales": len(dataset),
    "cantidad_registros_entrenamiento": int(dataset["foto_mes"].isin(train_final).sum()),
    "cantidad_registros_validacion": 0,
    "meses_entrenamiento": ",".join(map(str, train_final)),
    "meses_validacion": "",
    "columnas_excluidas": ",".join(columnas_excluir),
    "benchmark_original": benchmark_original,
}

display(pd.DataFrame([registro_validacion]))

,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
0,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,818414,0,"202103,202104,202105,202106,202107",,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-PSEUDO-001


## Guardar RDS y registrar

Ejecutar una vez por exportación. Genera un nuevo ID, su RDS y una fila en el CSV.

In [11]:
registro_guardado = guardar_rds_sin_validacion(params_completos, registro_validacion)
display(registro_guardado)

RDS: /home/marco/code/DMEyF/dmeyf2026/ensembles/man_val/53.rds


,id_validacion,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
0,53,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,818414,0,"202103,202104,202105,202106,202107",,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-PSEUDO-001


# Registro completo — últimas 10 filas

In [12]:
display(pd.read_csv(archivo_validaciones).tail(10))

,id_validacion,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
43,44,474831500.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
44,45,472384000.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lags.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
45,46,466537500.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lags.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_085726
46,47,465883000.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_085726
47,48,465883000.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_084709
48,49,466537500.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lags.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_084709
49,50,466537500.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lags.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_084709
50,51,474831500.0,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106.0,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
51,52,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,654066,0,"202103,202104,202105,202106",NaN,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-010
52,53,NaN,230047,competencia_01_ternaria_lf_k6_perdida4.csv,983061,818414,0,"202103,202104,202105,202106,202107",NaN,"ternaria,ternaria_lag1,clase01,fold,azar,train...",LAG-PSEUDO-001
